In [ ]:
import time
import csv
import os
import json
import base64
import requests

from pydantic import BaseModel, ConfigDict, Field
from typing import Optional
from datetime import datetime, date, timedelta


class OwnerModel(BaseModel):
    login: str
    type: str


class LicenseModel(BaseModel):
    spdx_id: Optional[str] = None

    model_config = ConfigDict(extra="ignore")


class RepositoryModel(BaseModel):
    github_id: int = Field(alias="id")
    full_name: str
    stargazers_count: int
    forks_count: int
    created_at: datetime
    pushed_at: Optional[datetime] = None
    topics: list[str] = []
    owner: OwnerModel
    subscribers_count: int = 0
    license: Optional[LicenseModel] = None

    model_config = ConfigDict(extra="ignore")


def decode_readme(content: str):
    decoded = base64.b64decode(content)
    return decoded.decode("utf-8", errors="replace")


class GitHubAPI:
    def __init__(
        self,
        token: str,
        sleep_between_requests: float = 0.2,
        max_retries: int = 5,
        rate_limit_buffer_seconds: int = 5,
    ):
        self.base_url = "https://api.github.com"
        self.headers = {
            "Accept": "application/vnd.github+json",
            "User-Agent": "GitHub-Research/1.0",
            "Authorization": f"Bearer {token}",
            "X-GitHub-Api-Version": "2022-11-28",
        }
        self.sleep_between_requests = sleep_between_requests
        self.max_retries = max_retries
        self.rate_limit_buffer_seconds = rate_limit_buffer_seconds
        self.session = requests.Session()
        self.session.headers.update(self.headers)

    def _request(self, method: str, url: str, **kwargs):
        last_exception = None

        for attempt in range(1, self.max_retries + 1):
            try:
                response = self.session.request(method, url, timeout=60, **kwargs)

                # rate limit
                if response.status_code == 403:
                    remaining = response.headers.get("X-RateLimit-Remaining")
                    reset_ts = response.headers.get("X-RateLimit-Reset")

                    if remaining == "0" and reset_ts:
                        reset_time = int(reset_ts)
                        now_ts = int(time.time())
                        wait_time = max(
                            reset_time - now_ts + self.rate_limit_buffer_seconds, 1
                        )

                        reset_dt = datetime.fromtimestamp(reset_time)
                        print(
                            f"rate limit hit for {url}. "
                            f"sleeping {wait_time} seconds until {reset_dt.isoformat()}"
                        )
                        time.sleep(wait_time)
                        continue

                # secondary limit / too many requests / temporary server errors
                if response.status_code in (429, 500, 502, 503, 504):
                    retry_after = response.headers.get("Retry-After")
                    if retry_after and retry_after.isdigit():
                        wait_time = int(retry_after)
                    else:
                        wait_time = min(2 ** attempt, 60)

                    print(
                        f"temporary error {response.status_code} for {url}. "
                        f"retrying in {wait_time} seconds (attempt {attempt}/{self.max_retries})"
                    )
                    time.sleep(wait_time)
                    continue

                response.raise_for_status()

                time.sleep(self.sleep_between_requests)
                return response

            except requests.RequestException as e:
                last_exception = e

                # retry only if attempts remain
                if attempt < self.max_retries:
                    wait_time = min(2 ** attempt, 60)
                    print(
                        f"request error for {url}: {e}. "
                        f"retrying in {wait_time} seconds (attempt {attempt}/{self.max_retries})"
                    )
                    time.sleep(wait_time)
                    continue

                break

        if last_exception is not None:
            raise last_exception

        raise RuntimeError(f"request failed after retries: {url}")

    def _get_count_from_link(self, url, params=None):
        response = self._request("GET", url, params=params)
        link = response.headers.get("Link", "")

        if 'rel="last"' in link:
            import re

            match = re.search(r"[?&]page=(\d+)>; rel=\"last\"", link)
            if match:
                return int(match.group(1))

        data = response.json()
        if isinstance(data, list):
            return len(data)
        return 0

    def search_repo(self, query: str, page: int = 1, per_page: int = 100):
        url = f"{self.base_url}/search/repositories"
        params = {"q": query, "page": page, "per_page": per_page}
        response = self._request("GET", url, params=params)
        data = response.json()
        return [RepositoryModel.model_validate(repo) for repo in data["items"]]

    def get_repo(self, owner: str, repo: str):
        url = f"{self.base_url}/repos/{owner}/{repo}"
        response = self._request("GET", url)
        return RepositoryModel.model_validate(response.json())

    def get_readme(self, owner: str, repo: str):
        url = f"{self.base_url}/repos/{owner}/{repo}/readme"
        response = self._request("GET", url)
        data = response.json()
        content = data.get("content", "")
        return decode_readme(content)

    def get_releases_count(self, owner: str, repo: str):
        url = f"{self.base_url}/repos/{owner}/{repo}/releases"
        return self._get_count_from_link(url, {"per_page": 1})

    def get_contributors_count(self, owner: str, repo: str):
        url = f"{self.base_url}/repos/{owner}/{repo}/contributors"
        return self._get_count_from_link(url, {"per_page": 1, "anon": "true"})

    def get_languages(self, owner: str, repo: str):
        url = f"{self.base_url}/repos/{owner}/{repo}/languages"
        response = self._request("GET", url)
        data = response.json()
        total = sum(data.values())
        if total == 0:
            return {}

        return {
            language: round(bytes_count / total * 100, 2)
            for language, bytes_count in data.items()
        }

    def get_issues_count(self, owner: str, repo: str, is_closed: bool):
        state = "closed" if is_closed else "open"
        url = f"{self.base_url}/search/issues"
        params = {"q": f"repo:{owner}/{repo} type:issue state:{state}"}
        response = self._request("GET", url, params=params)
        return response.json()["total_count"]

    def get_pr_count(self, owner: str, repo: str, is_closed: bool):
        state = "closed" if is_closed else "open"
        url = f"{self.base_url}/search/issues"
        params = {"q": f"repo:{owner}/{repo} type:pr state:{state}"}
        response = self._request("GET", url, params=params)
        return response.json()["total_count"]

    def get_commits_count(self, owner: str, repo: str):
        url = f"{self.base_url}/repos/{owner}/{repo}/commits"
        return self._get_count_from_link(url, {"per_page": 1})

    def get_owner_location(self, owner: OwnerModel):
        if owner.type == "User":
            url = f"{self.base_url}/users/{owner.login}"
        elif owner.type == "Organization":
            url = f"{self.base_url}/orgs/{owner.login}"
        else:
            return None

        response = self._request("GET", url)
        data = response.json()
        return data.get("location")


def daterange(start_date: date, end_date: date):
    current = start_date
    while current <= end_date:
        yield current
        current += timedelta(days=1)


def safe_call(func, *args, default=None, errors=(requests.HTTPError, Exception), **kwargs):
    try:
        return func(*args, **kwargs)
    except errors as e:
        print(f"warning: {func.__name__} failed for args={args}: {e}")
        return default


def search_repositories_for_day(
    github: GitHubAPI,
    target_date: date,
    min_stars: int = 1000,
    per_page: int = 100,
    max_pages: int = 10,
):
    query = f"created:{target_date.isoformat()} stars:>{min_stars}"
    repos = []

    for page in range(1, max_pages + 1):
        items = github.search_repo(query=query, page=page, per_page=per_page)
        if not items:
            break

        repos.extend(items)

        if len(items) < per_page:
            break

    return repos


def build_repository_record(github: GitHubAPI, repo_item: RepositoryModel) -> dict:
    owner = repo_item.owner.login
    repo_name = repo_item.full_name.split("/", 1)[1]

    full_repo = safe_call(github.get_repo, owner, repo_name)
    readme = safe_call(github.get_readme, owner, repo_name, default=None)
    releases_count = safe_call(github.get_releases_count, owner, repo_name, default=0)
    contributors_count = safe_call(github.get_contributors_count, owner, repo_name, default=0)
    languages_map = safe_call(github.get_languages, owner, repo_name, default={})
    open_issues_count = safe_call(github.get_issues_count, owner, repo_name, False, default=0)
    closed_issues_count = safe_call(github.get_issues_count, owner, repo_name, True, default=0)
    open_pr_count = safe_call(github.get_pr_count, owner, repo_name, False, default=0)
    closed_pr_count = safe_call(github.get_pr_count, owner, repo_name, True, default=0)
    commits_count = safe_call(github.get_commits_count, owner, repo_name, default=0)
    owner_location = safe_call(github.get_owner_location, repo_item.owner, default=None)

    return {
        "github_id": repo_item.github_id,
        "readme": readme,
        "releases_count": releases_count,
        "subscribers_count": full_repo.subscribers_count if full_repo else 0,
        "stargazers_count": repo_item.stargazers_count,
        "forks_count": repo_item.forks_count,
        "created_at": repo_item.created_at.isoformat() if repo_item.created_at else None,
        "license_spdx_id": (
            full_repo.license.spdx_id if full_repo and full_repo.license else None
        ),
        "topics": repo_item.topics,
        "pushed_at": repo_item.pushed_at.isoformat() if repo_item.pushed_at else None,
        "languages_map": languages_map,
        "open_issues_count": open_issues_count,
        "closed_issues_count": closed_issues_count,
        "open_pr_count": open_pr_count,
        "closed_pr_count": closed_pr_count,
        "full_name": repo_item.full_name,
        "contributors_count": contributors_count,
        "commits_count": commits_count,
        "owner_location": owner_location,
    }


FIELDNAMES = [
    "github_id",
    "readme",
    "releases_count",
    "subscribers_count",
    "stargazers_count",
    "forks_count",
    "created_at",
    "license_spdx_id",
    "topics",
    "pushed_at",
    "languages_map",
    "open_issues_count",
    "closed_issues_count",
    "open_pr_count",
    "closed_pr_count",
    "full_name",
    "contributors_count",
    "commits_count",
    "owner_location",
]


def normalize_row_for_csv(row: dict) -> dict:
    row = dict(row)
    row["topics"] = json.dumps(row.get("topics", []), ensure_ascii=False)
    row["languages_map"] = json.dumps(row.get("languages_map", {}), ensure_ascii=False)
    return row


def append_row_to_csv(file_path: str, row: dict, fieldnames: list[str]):
    file_exists = os.path.exists(file_path)
    normalized_row = normalize_row_for_csv(row)

    with open(file_path, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)

        if not file_exists or os.path.getsize(file_path) == 0:
            writer.writeheader()

        writer.writerow(normalized_row)


def load_existing_github_ids(file_path: str) -> set[int]:
    existing_ids = set()

    if not os.path.exists(file_path) or os.path.getsize(file_path) == 0:
        return existing_ids

    with open(file_path, "r", newline="", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            github_id = row.get("github_id")
            if github_id:
                try:
                    existing_ids.add(int(github_id))
                except ValueError:
                    pass

    return existing_ids


def collect_repositories_to_csv(
    token: str,
    year: int,
    output_file: str,
    start_from: Optional[date] = None,
    end_at: Optional[date] = None,
    min_stars: int = 1000,
    sleep_between_requests: float = 0.2,
):
    github = GitHubAPI(token=token, sleep_between_requests=sleep_between_requests)

    start_date = start_from or date(year, 1, 1)
    end_date = end_at or date(year, 12, 31)

    seen_ids = load_existing_github_ids(output_file)
    print(f"already saved repositories: {len(seen_ids)}")

    for day in daterange(start_date, end_date):
        print(f"\nSearch for {day.isoformat()} ...")

        try:
            repos = search_repositories_for_day(
                github=github,
                target_date=day,
                min_stars=min_stars,
                per_page=100,
                max_pages=10,
            )
        except Exception as e:
            print(f"search failed for {day.isoformat()}: {e}")
            continue

        print(f"  found: {len(repos)}")

        for repo_item in repos:
            if repo_item.github_id in seen_ids:
                print(f"    skip existing: {repo_item.full_name}")
                continue

            print(f"    -> collecting {repo_item.full_name}")

            try:
                record = build_repository_record(github, repo_item)
                append_row_to_csv(output_file, record, FIELDNAMES)
                seen_ids.add(repo_item.github_id)
                print(f"       saved: {repo_item.full_name}")
            except Exception as e:
                print(f"       error while saving {repo_item.full_name}: {e}")


if __name__ == "__main__":
    GITHUB_TOKEN = ""
    
    collect_repositories_to_csv(
        token=GITHUB_TOKEN,
        year=2025,
        output_file="github_repositories_2025.csv",
        start_from=date(2025, 1, 1),
        end_at=date(2025, 12, 31),
        min_stars=1000,
        sleep_between_requests=0.15,
    )